In [113]:
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [114]:
class BoostedTree:
  def __init__(self,X,gradients,hessians,min_child_weight,gamma,reg_lambda,max_depth,ids=None):
    self.X = X # X is expected to be a numpy array
    self.gradients = gradients
    self.hessians = hessians
    self.min_child_weight = min_child_weight
    self.gamma = gamma
    self.reg_lambda = reg_lambda
    self.max_depth = max_depth
    self.req_ids=ids if ids is not None else np.arange(len(gradients))
    self.cnt_feature=X.shape[1]
    self.opt_weight=-1*((self.gradients[self.req_ids].sum())/(self.hessians[self.req_ids].sum()+self.reg_lambda))
    self.threshold=0.0
    self.split_score=0.0
    self.split_ids=0
    self.build_tree()

  def build_tree(self):
    # building the tree recursively after each step finding the best splits, if a valid split is not found, it becomes leaf node
    if self.max_depth<=0:
      return

    for i in range(self.cnt_feature):
      self.find_best_split(i)

    if self.split_score<=0.0:  # no valid split, this will be leaf node
      return

    # partitioning the data based on best split
    X_col=self.X[self.req_ids,self.split_ids]
    left_ids=self.req_ids[X_col<=self.threshold]
    right_ids=self.req_ids[X_col>self.threshold]

    # Building subtrees using recursion
    self.left=BoostedTree(
        self.X,self.gradients,self.hessians,
        self.min_child_weight,self.gamma,self.reg_lambda,
        self.max_depth-1,ids=left_ids
    )

    self.right=BoostedTree(
        self.X,self.gradients,self.hessians,
        self.min_child_weight,self.gamma,self.reg_lambda,
        self.max_depth-1,ids=right_ids
    )


  def find_best_split(self,idx):
    # now let's find the best split for a particular given feature
    X_col=self.X[self.req_ids,idx]
    G=self.gradients[self.req_ids]
    H=self.hessians[self.req_ids]

    #sorting by feature value
    sorted_ids=np.argsort(X_col)
    X_col=X_col[sorted_ids]
    G=G[sorted_ids]
    H=H[sorted_ids]

    G_total=G.sum()
    H_total=H.sum()

    left_G=0.0
    left_H=0.0

    best_gain=-float('inf')
    best_tresh=None
    best_index=None

    for i in range(1,len(X_col)):
      left_G+=G[i-1]
      left_H+=H[i-1]
      right_G=G_total-left_G
      right_H=H_total-left_H

      if X_col[i]==X_col[i-1]:
        continue

      gain=0.5*((left_G**2)/(left_H+self.reg_lambda)+(right_G **2)/(right_H+self.reg_lambda)-(G_total**2)/(H_total+self.reg_lambda))-self.gamma

      if gain>best_gain:
        best_gain=gain
        best_tresh=(X_col[i]+X_col[i-1])/2.0
        best_index=i

    if best_gain>self.split_score:
      self.split_score=best_gain
      self.threshold=best_tresh
      self.split_ids=idx

  def predict(self,X):
    # predicting for all sample in X
    return np.array([self._predict_single(x) for x in X])

  def _predict_single(self,x):
    # predicting for a single sample
    if self.split_score<=0.0:
      return self.opt_weight

    if x[self.split_ids]<=self.threshold:
      return self.left._predict_single(x)
    else:
      return self.right._predict_single(x)

In [115]:
class XGBoostRegressor:
  def __init__(self,n_estimators=100,learning_rate=0.1,max_depth=3,min_child_weight=1.0,gamma=0.0,reg_lambda=1.0):
    self.n_estimators=n_estimators
    self.learning_rate=learning_rate
    self.max_depth=max_depth
    self.min_child_weight=min_child_weight
    self.gamma=gamma
    self.reg_lambda=reg_lambda
    self.trees=[]

  def fit(self,X,y):
    # intial prediction (y_hat) is zero
    y_pred=np.zeros(len(y))

    # Ensure X is a numpy array
    X_np = X if isinstance(X, np.ndarray) else X.to_numpy()


    for i in range(self.n_estimators):
      # computing gradient and hessian for mse
      grad=y_pred-y  # first derivate of (1/2)(y-y_pred)^2
      hess=np.ones_like(y)  # second derivative is 1

      tree=BoostedTree(
          X_np,grad,hess,self.min_child_weight,self.gamma,self.reg_lambda,self.max_depth
      )
      # updating predictions
      y_pred+=self.learning_rate*tree.predict(X_np)
      self.trees.append(tree)

  def predict(self,X):
    # predicting for all samples
    y_pred=np.zeros(len(X))
    # Ensure X is a numpy array for prediction
    X_np = X if isinstance(X, np.ndarray) else X.to_numpy()
    for tree in self.trees:
      y_pred+=self.learning_rate*tree.predict(X_np)
    return y_pred

In [116]:
# Example usage
X=np.array([[1],[2],[3],[4],[5]])
y=np.array([1.1,1.9,3.0,3.9,5.2])

model=XGBoostRegressor(
    n_estimators=1000,
    learning_rate=0.1,
    max_depth=3,
    min_child_weight=1,
    gamma=0.0,
    reg_lambda=1.0
)
model.fit(X,y)
preds = model.predict(X)
print("Predictions:", preds)

Predictions: [1.1 1.9 3.  3.9 5.2]


In [117]:
data=pd.read_csv('Fish.csv')
data=pd.get_dummies(data,columns=["Species"])
data=data.astype(float)
data

,Weight,Length1,Length2,Length3,Height,Width,Species_Bream,Species_Parkki,Species_Perch,Species_Pike,Species_Roach,Species_Smelt,Species_Whitefish
0,242.0,23.2,25.4,30.0,11.5200,4.0200,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,290.0,24.0,26.3,31.2,12.4800,4.3056,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,340.0,23.9,26.5,31.1,12.3778,4.6961,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,363.0,26.3,29.0,33.5,12.7300,4.4555,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,430.0,26.5,29.0,34.0,12.4440,5.1340,1.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
154,12.2,11.5,12.2,13.4,2.0904,1.3936,0.0,0.0,0.0,0.0,0.0,1.0,0.0
155,13.4,11.7,12.4,13.5,2.4300,1.2690,0.0,0.0,0.0,0.0,0.0,1.0,0.0
156,12.2,12.1,13.0,13.8,2.2770,1.2558,0.0,0.0,0.0,0.0,0.0,1.0,0.0
157,19.7,13.2,14.3,15.2,2.8728,2.0672,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [118]:
X=data.drop(columns=['Weight']).values
y=data['Weight'].values

In [119]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.1,random_state=42)

In [120]:
y_test

array([  78. ,   13.4,  200. ,  270. ,  150. , 1000. ,    7. ,  180. ,
        188. , 1250. ,  650. , 1000. ,  600. ,  150. ,  700. ,  920. ])

In [121]:
model=XGBoostRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    min_child_weight=1,
    gamma=0.0,
    reg_lambda=1
)
model.fit(X_train,y_train)
preds = model.predict(X_test)
print("Predictions:", preds)

Predictions: [  80.52671073   13.52225558  281.4101215   263.94494676  141.80105939
  912.80742974   12.34740755  224.10297598  172.91973375 1225.98242058
  673.8089893   989.45848914  579.16036614  107.14393413  691.04556205
  940.68947119]


In [122]:
from sklearn.metrics import mean_squared_error

mse=mean_squared_error(y_test,preds)
print(mse)

1285.9702394710862


In [127]:
from xgboost import XGBRegressor
model = XGBRegressor(n_estimators=200,learning_rate=0.05)
model.fit(X_train,y_train)
preds1 = model.predict(X_test)
print("Predictions:", preds1)

Predictions: [  83.80622     12.242655   295.7269     269.10132    145.19444
  899.9961       7.8902698  180.83763    177.43575   1037.5314
  676.4924     910.26416    583.44727    112.33718    698.61523
  951.0966   ]


In [128]:
from sklearn.metrics import mean_squared_error

mse=mean_squared_error(y_test,preds1)
print(mse)

4743.44708533748
